In [ ]:
!pip install -q unsloth
# !pip install "numpy<2" --force-reinstall

In [ ]:
from unsloth import FastLanguageModel

# 1. Point this to the folder you saved yesterday
save_directory = "/kaggle/input/models/pritepakorn/mt-jp-to-thai/other/default/1/Qwen3-14B-unsloth-bnb-4bit"
# save_directory = "/kaggle/input/models/pritepakorn/mt-jp-to-thai/other/default/1/Qwen3-14B-Base-unsloth-bnb-4bit"

# 2. Load the model with your custom LoRA adapter applied
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = save_directory, # Unsloth automatically finds the base model and attaches your adapter
    max_seq_length = 2048,
    # dtype = None,
    load_in_4bit = True,
)

# 3. Put it in inference mode for fast generation
FastLanguageModel.for_inference(model)

print("Custom Manga Translator Model loaded and ready for inference!")

In [ ]:
!pip install -q pythainlp sacrebleu unbabel-comet google-genai

In [ ]:
from datasets import load_dataset, DatasetDict

dataset = load_dataset("csv", data_files="/kaggle/input/practical-project-dataset/jp_to_thai_mapping.csv")

# 2. Perform the first split: 80% Train, 20% Test+Validation
train_testvalid = dataset["train"].train_test_split(test_size=0.2, seed=42)

# 3. Perform the second split: Divide the 20% equally into Validation (10%) and Test (10%)
test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=42)

# 4. Combine them into a single, organized DatasetDict
split_dataset = DatasetDict({
    "train": train_testvalid["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
})

In [ ]:
from tqdm.auto import tqdm

# We will store the results here
sources = []
references = []
predictions = []

print("Generating translations for the test set...")
for row in tqdm(split_dataset["test"]):
    jp_text = row["japanese"]
    thai_truth = row["thai"]
    
    # Format the prompt using the same structure as training
    messages = [
        {"role": "system", "content": "You are an expert manga translator. Translate the Japanese text into natural, fluent Thai. Ensure the translation captures the original tone, emotion, and character nuances."},
        {"role": "user", "content": f"Japanese: {jp_text}"}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    
    # Generate the prediction
    outputs = model.generate(**inputs, max_new_tokens=256, temperature = 0.7, top_p = 0.8, top_k = 20)
    
    # Decode only the new generated tokens (skip the prompt)
    prompt_length = inputs['input_ids'].shape[1]
    pred_text = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
    
    sources.append(jp_text)
    references.append(thai_truth)
    predictions.append(pred_text)

In [ ]:
import pandas as pd

# 1. Combine your lists into a dictionary
results_dict = {
    "Japanese_Source": sources,
    "Thai_Reference": references,
    "Model_Prediction": predictions
}

# 2. Convert the dictionary into a Pandas DataFrame
df_results = pd.DataFrame(results_dict)

# 3. Save to CSV
# Using 'utf-8-sig' is highly recommended for Thai text so it displays correctly if opened in Excel
df_results.to_csv("/kaggle/working/translation_results.csv", index=False, encoding="utf-8-sig")

print("Successfully saved translations to translation_results.csv!")

# Display the first few rows to verify
display(df_results.head())

In [ ]:
import torch
import gc

# 1. Delete the Unsloth model and tokenizer variables
del model
del tokenizer

# 2. Force Python to garbage-collect the deleted objects
gc.collect()

# 3. Empty the PyTorch CUDA cache so the GPU actually frees up the VRAM
torch.cuda.empty_cache()

print("GPU Memory cleared!")

In [ ]:
from comet import download_model, load_from_checkpoint

print("Loading COMET Model (This may take a minute to download)...")
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

# Prepare data for COMET
comet_data = [
    {"src": src, "mt": pred, "ref": ref} 
    for src, pred, ref in zip(sources, predictions, references)
]

print("Calculating COMET Score...")
# COMET automatically utilizes your GPU for fast evaluation
comet_result = comet_model.predict(comet_data, batch_size=8, gpus=1)

# The score is usually between 0 and 1. Multiply by 100 for readability.
print(f"Average COMET Score: {comet_result.system_score * 100:.2f}")

In [ ]:
import sacrebleu
from pythainlp.tokenize import word_tokenize

def thai_tokenizer(text):
    # Uses PyThaiNLP's 'newmm' dictionary-based word segmentation
    return " ".join(word_tokenize(str(text), engine="newmm"))

print("Calculating BLEU Score...")
# Tokenize both references and predictions
refs_tok = [[thai_tokenizer(ref) for ref in references]]
preds_tok = [thai_tokenizer(pred) for pred in predictions]

# Calculate BLEU
bleu = sacrebleu.corpus_bleu(preds_tok, refs_tok)
print(f"Thai-Tokenized BLEU Score: {bleu.score:.2f}")

In [ ]:
import os
import json
import time
import pandas as pd
from tqdm.auto import tqdm
from google import genai
from google.genai import types

# 1. Define your API keys and the rotation logic
API_KEYS = [
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
    "REDACTED_API_KEY",
]
API_KEY_INDEX = 0

def _get_next_api_key():
    global API_KEY_INDEX
    if not API_KEYS:
        raise ValueError("API_KEYS list is empty. Please add your keys.")
    
    key = API_KEYS[API_KEY_INDEX % len(API_KEYS)]
    API_KEY_INDEX += 1
    return key

# 2. Define the scoring function with INFINITE Retry logic
def get_gemini_score(jp_text, ref_text, pred_text):
    model = "gemini-2.5-flash" 
    
    prompt = f"""You are an expert manga translation evaluator.
    Rate the AI Translation against the Reference Translation based on accuracy, fluency, and keeping the manga tone intact.
    
    Source (Japanese): {jp_text}
    Reference Translation (Thai): {ref_text}
    AI Translation (Thai): {pred_text}
    
    Return ONLY a valid JSON object with a single key "score" containing an integer from 1 to 10."""

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt)],
        ),
    ]

    generate_content_config = types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.0, 
    )

    attempt = 0
    # --- INFINITE RETRY LOOP ---
    while True:
        api_key = _get_next_api_key()
        client = genai.Client(api_key=api_key)
        
        try:
            response = client.models.generate_content(
                model=model,
                contents=contents,
                config=generate_content_config,
            )
            
            result = json.loads(response.text)
            
            # Ensure the API didn't return an empty JSON before accepting it
            if "score" in result:
                return result["score"]
            else:
                raise ValueError("Valid JSON returned, but 'score' key is missing.")
            
        except Exception as e:
            attempt += 1
            # Cap the wait time at 60 seconds so it doesn't sleep forever on long outages
            wait_time = min(5 * attempt, 60) 
            print(f"\n[Warning] Attempt {attempt} failed with key ...{api_key[-4:]}: {e}")
            print(f"Waiting {wait_time} seconds before retrying...")
            time.sleep(wait_time)

# 3. Resume Logic & Setup
csv_filename = "manga_translation_evaluation_results.csv"
start_index = 0

# Check if the file already exists to resume progress
if os.path.exists(csv_filename):
    try:
        existing_df = pd.read_csv(csv_filename)
        start_index = len(existing_df)
        print(f"Found existing save file! Resuming from row {start_index} out of {len(sources)}...")
    except pd.errors.EmptyDataError:
        print("Existing CSV is empty. Starting from the beginning...")

# 4. The Evaluation Loop (Saving every row)
print("Calculating Gemini Evaluation Scores...")

# (Assuming 'sources', 'references', and 'predictions' lists are already populated)
# Start the loop from where we left off, and keep the progress bar accurate
for i in tqdm(range(start_index, len(sources)), initial=start_index, total=len(sources)):
    src = sources[i]
    ref = references[i]
    pred = predictions[i]
    
    score = get_gemini_score(src, ref, pred)
    
    # Create a DataFrame for just this single row
    row_df = pd.DataFrame([{
        "Japanese Source": src,
        "Real Thai (Reference)": ref,
        "AI Thai (Prediction)": pred,
        "Gemini Score": score
    }])
    
    # Append to CSV. If it's the very first row ever, write the header. Otherwise, skip header.
    write_header = not os.path.exists(csv_filename)
    row_df.to_csv(csv_filename, mode='a', index=False, header=write_header, encoding="utf-8-sig")
    
    # Standard sleep to pace requests
    time.sleep(2) 

# 5. Final Calculation
print("\nEvaluation Complete!")
final_df = pd.read_csv(csv_filename)

valid_scores = [score for score in final_df["Gemini Score"] if score > 0]
average_gemini = sum(valid_scores) / len(valid_scores) if valid_scores else 0

print(f"Final Average Gemini Manga Score: {average_gemini:.2f} / 10")